![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# Transformers and Hugging Face
*Data Science & Machine Learning - Week 7, Day 3*

## How to use this notebook

Yesterday's sentence embeddings came out of a small **transformer**, and you used it as a black box: text in, vector out. Today you open the box, and you learn to use the thousands of pretrained transformers on the **Hugging Face Hub**, the place where the community shares models.

You will:

1. See how a transformer reads text: **tokens**, ids and attention masks.
2. Get **contextual vectors** from a pretrained model, and see how *bank* changes with the sentence around it.
3. Watch **masked language modelling**, the game BERT-style models play to learn language.
4. Use ready-made models with `pipeline`: sentiment, named entities and **zero-shot** classification.
5. **Fine-tune** DistilBERT on Day 1's forum posts and compare it with TF-IDF and yesterday's embeddings.
6. Meet a small **generative** model, and see next-token prediction, the idea behind the large language models of Days 4 and 5.

The slides **1. Language Models** in this folder go with the notebook (there is a PDF copy too). Run the cells in order. The **Appendix** has extra material that is not part of the lesson.

## Table of contents

- [Setup](#Setup)
- [1. Why transformers](#1.-Why-transformers)
- [2. Tokens: how a transformer reads text](#2.-Tokens:-how-a-transformer-reads-text)
- [3. Contextual vectors](#3.-Contextual-vectors)
- [4. How BERT learns: masked language modelling](#4.-How-BERT-learns:-masked-language-modelling)
- [5. Ready-made models with pipeline](#5.-Ready-made-models-with-pipeline)
    - [Sentiment analysis](#Sentiment-analysis)
    - [Named entities](#Named-entities)
    - [Zero-shot classification](#Zero-shot-classification)
- [6. Fine-tuning DistilBERT](#6.-Fine-tuning-DistilBERT)
    - [Which method wins?](#Which-method-wins?)
- [7. Generative models: predicting the next token](#7.-Generative-models:-predicting-the-next-token)
- [8. Choosing a model responsibly](#8.-Choosing-a-model-responsibly)
- [Summary](#Summary)
- [Appendix](#Appendix)

## Setup

You need `transformers`, `torch`, `datasets` and `accelerate`, plus `sentence-transformers` from yesterday. If the import cell fails, install them once from a terminal with `pip install transformers torch datasets accelerate sentence-transformers`.

The models are downloaded the first time you use them and cached in `~/.cache/huggingface`. Together they take about 1.3 GB, so the first run needs a good connection and a few minutes. Everything runs on a normal laptop CPU, with no account and no API key. Fine-tuning in section 6 takes one to four minutes, depending on your computer.

In [1]:
import os
import logging
import warnings
os.environ["HF_HUB_VERBOSITY"] = "error"      # hide Hugging Face download notices
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*torch.jit.script.*")

import time
import numpy as np
import pandas as pd
import torch

from transformers import (pipeline, AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
                          AutoModelForCausalLM, TrainingArguments, Trainer)
from transformers.utils import logging as transformers_logging
transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()
import datasets
datasets.disable_progress_bars()
from sentence_transformers import SentenceTransformer

from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score

pd.set_option("display.max_colwidth", 110)
_ = torch.manual_seed(0)

## 1. Why transformers

Over the last two days you saw three ways to turn text into numbers:

+ **Bag-of-Words and TF-IDF** (Day 1) count words. They know nothing about meaning, so *car* and *automobile* are unrelated.
+ **Word vectors** (Day 2) give each word one fixed vector learned from huge amounts of text. *Car* and *automobile* are close, but *bank* has the same vector by a river and in the city.
+ **Sentence embeddings** (Day 2) read the whole sentence, so context and word order count.

The last one is a transformer, and transformers are behind almost every NLP system built since 2018. Three ideas make them powerful.

**Attention.** When the model builds the vector of a word, it looks at every other word in the text and decides how much each one matters. In *I sat on the river bank*, the word *bank* pays a lot of attention to *river*, so its vector ends up about rivers. A transformer repeats this in many **layers**, so the vectors get more and more informed by the context.

[visual: the sentence "I sat on the river bank" with arrows from "bank" to every other word, the arrow to "river" much thicker than the others]

**Size.** Attention can be computed for all the words at once, so transformers train very efficiently on modern hardware. That made it possible to train them on billions of words, with millions or billions of parameters.

**Transfer learning.** A model is first **pretrained** once, at great cost, on a simple task that needs no labels, such as filling in hidden words. This gives it general *language knowledge*. Then anyone can **fine-tune** it on a small labelled dataset to give it *task knowledge*, such as telling hockey posts from space posts. You saw the same idea with pretrained image models in Week 5.

[visual: two stages. Stage 1, pretraining: a huge pile of unlabelled text → model learns language, weeks on hundreds of GPUs. Stage 2, fine-tuning: a few hundred labelled examples → the same model learns a task, minutes on a laptop]

There are two big families, and you use both today:

| Family | Reads the text | Pretrained to | Good at | Examples |
|---|---|---|---|---|
| **Encoders** | all at once, in both directions | fill in hidden words | understanding: classification, entities, embeddings | BERT, DistilBERT, MiniLM |
| **Decoders** | left to right | predict the next word | generating text | GPT, Llama, SmolLM |

The **Hugging Face** `transformers` library gives the same few commands for all of them, and the [Hub](https://huggingface.co/models) hosts more than a million pretrained models.

## 2. Tokens: how a transformer reads text

A transformer does not read words. It reads **tokens**, pieces of words from a fixed vocabulary, each with an id. Every model comes with its own **tokenizer**, and you must always use the tokenizer that matches the model.

`AutoTokenizer.from_pretrained` downloads the right tokenizer for a model name from the Hub. The model here is **DistilBERT**, a smaller and faster version of BERT (66 million parameters, 40% smaller than BERT and about as accurate):

In [2]:
checkpoint = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

print("Vocabulary size:", tokenizer.vocab_size)
print(tokenizer.tokenize("Transformers read tokens, not words."))

Vocabulary size: 30522
['transformers', 'read', 'token', '##s', ',', 'not', 'words', '.']


The vocabulary has about 30,000 tokens. Common words are a single token, and the text is lowercased, because this is the *uncased* version of the model. Now look at words that GloVe did not know yesterday:

In [3]:
for text in ["chatgpt", "tokenization", "unbelievably", "Banking77"]:
    print(f"{text:<14} {tokenizer.tokenize(text)}")

chatgpt        ['chat', '##gp', '##t']
tokenization   ['token', '##ization']
unbelievably   ['un', '##bel', '##ie', '##va', '##bly']
Banking77      ['banking', '##7', '##7']


Rare words are split into known pieces. The `##` prefix means "this piece continues the previous one". This is called **subword tokenization**, and it means that no word is ever unknown: *chatgpt*, which did not exist when DistilBERT was trained, becomes `chat`, `##gp` and `##t`. The model may not know what it means, but it can still read it.

Calling the tokenizer directly gives what the model actually receives:

In [4]:
encoded = tokenizer("My card has not arrived.")
print("input_ids:     ", encoded["input_ids"])
print("tokens:        ", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print("attention_mask:", encoded["attention_mask"])

input_ids:      [101, 2026, 4003, 2038, 2025, 3369, 1012, 102]
tokens:         ['[CLS]', 'my', 'card', 'has', 'not', 'arrived', '.', '[SEP]']
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1]


Three things to notice:

+ **`input_ids`** are the token ids, the only thing the model sees.
+ Two **special tokens** were added: `[CLS]` at the start and `[SEP]` at the end. BERT-style models were trained with them, and the vector of `[CLS]` is often used to represent the whole text.
+ **`attention_mask`** says which tokens are real (1) and which are padding (0). Here all of them are real.

Models process texts in **batches**, and every text in a batch must have the same length. `padding=True` fills the shorter ones with `[PAD]` tokens, and `truncation=True` with `max_length` cuts the longer ones:

In [5]:
batch = tokenizer(["Where is my card?", "I ordered a new card two weeks ago and it still has not arrived."],
                  padding=True, truncation=True, max_length=12, return_tensors="pt")
print(batch["input_ids"])
print(batch["attention_mask"])

tensor([[ 101, 2073, 2003, 2026, 4003, 1029,  102,    0,    0,    0,    0,    0],
        [ 101, 1045, 3641, 1037, 2047, 4003, 2048, 3134, 3283, 1998, 2009,  102]])
tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


The first text got five `0` ids (the `[PAD]` token) and zeros in its attention mask, so the model will ignore those positions. The second text was cut to 12 tokens. `return_tensors="pt"` returns PyTorch tensors, the format the model expects.

Every model has a maximum length: 512 tokens for DistilBERT, 256 for yesterday's MiniLM. Anything longer is cut, which is why long documents are split into chunks for search (Day 5).

## 3. Contextual vectors

`AutoModel` loads the pretrained transformer without any task on top. Give it the tokenizer's output and it returns one vector per token, from its last layer:

In [6]:
encoder = AutoModel.from_pretrained(checkpoint)

inputs = tokenizer("I deposited cash at the bank.", return_tensors="pt")
with torch.no_grad():                    # no training here, so no gradients needed
    outputs = encoder(**inputs)

print("Tokens:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))
print("Shape of last_hidden_state:", tuple(outputs.last_hidden_state.shape))

Tokens: ['[CLS]', 'i', 'deposited', 'cash', 'at', 'the', 'bank', '.', '[SEP]']
Shape of last_hidden_state: (1, 9, 768)


The shape is (batch, tokens, numbers per token): one text, 9 tokens, and 768 numbers for each token. `torch.no_grad()` tells PyTorch that you are not training, which saves memory and time.

These vectors are **contextual**: the same word gets a different vector in each sentence. Take the vector of *bank* in three sentences and compare them:

In [7]:
def token_vector(sentence, word):
    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        hidden = encoder(**inputs).last_hidden_state[0]
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    return hidden[tokens.index(word)].numpy()

sentences = ["I deposited cash at the bank.", "The bank approved my loan.", "We had a picnic on the river bank."]
vectors = np.vstack([token_vector(s, "bank") for s in sentences])
vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
pd.DataFrame((vectors @ vectors.T).round(2), index=sentences, columns=["cash", "loan", "river"])

,cash,loan,river
I deposited cash at the bank.,1.00,0.91,0.69
The bank approved my loan.,0.91,1.00,0.63
We had a picnic on the river bank.,0.69,0.63,1.00


The two money meanings of *bank* score 0.91, and the river bank is clearly further away (0.69 and 0.63). Yesterday, GloVe gave *bank* one vector whatever the sentence. A transformer gives it a different vector each time, because attention mixes in the words around it.

This is the slides' example of *capital*: the vector of *capital* in *capital city* ends up closer to geography than to finance or crime.

## 4. How BERT learns: masked language modelling

How did DistilBERT learn all this without anyone labelling anything? During pretraining, about 15% of the words in each sentence were hidden behind a `[MASK]` token, and the model had to guess them. After billions of guesses on Wikipedia and books, it got very good at it, and to be good at it, it had to learn grammar, meaning and a lot of facts about the world.

[visual: "Capital [MASK], also known as the death [MASK]" → BERT → "punishment", "penalty"]

The `fill-mask` pipeline lets you play the same game with the pretrained model:

In [8]:
fill = pipeline("fill-mask", model=checkpoint)

for sentence in ["Paris is the capital of [MASK].",
                 "I forgot my [MASK], so I cannot log in to the app.",
                 "The astronauts returned to [MASK] after six months in orbit."]:
    guesses = fill(sentence, top_k=4)
    print(sentence, [(g["token_str"], round(g["score"], 2)) for g in guesses])

Paris is the capital of [MASK]. [('france', 0.49), ('martinique', 0.1), ('algeria', 0.04), ('haiti', 0.03)]
I forgot my [MASK], so I cannot log in to the app. [('password', 0.11), ('iphone', 0.06), ('email', 0.05), ('phone', 0.05)]
The astronauts returned to [MASK] after six months in orbit. [('earth', 0.5), ('orbit', 0.2), ('mars', 0.07), ('space', 0.02)]


*France*, *password* and *earth* are the top guesses. The model knows facts (Paris is in France), common situations (passwords and logging in) and what astronauts do. It is not perfect, and the other guesses show it: *martinique* and *algeria* for the capital, *iphone* for the login. And it is only playing a game: it fills in a word, it does not answer questions or classify texts.

That is the limit the slides point out. Masked language modelling gives **language knowledge**, but no **task knowledge**. To classify texts, you either use a model that someone already fine-tuned for your task (section 5) or fine-tune one yourself (section 6).

## 5. Ready-made models with pipeline

Thousands of models on the Hub have already been fine-tuned for a task. `pipeline` is the fastest way to use them: it does the tokenization, runs the model and turns its output into something readable, in one call.

[visual: text → tokenizer → model → post-processing (scores to labels) → result, all three boxes inside one box labelled "pipeline"]

### Sentiment analysis

This DistilBERT was fine-tuned on SST-2, a dataset of film reviews labelled positive or negative:

In [9]:
sentiment = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

reviews = ["I waited two hours and nobody helped me.",
           "The service was not bad at all.",
           "Great, my card was declined again.",
           "The app is okay."]
pd.DataFrame([{"text": r, **sentiment(r)[0]} for r in reviews])

,text,label,score
0,I waited two hours and nobody helped me.,NEGATIVE,0.999572
1,The service was not bad at all.,POSITIVE,0.996990
2,"Great, my card was declined again.",NEGATIVE,0.967879
3,The app is okay.,POSITIVE,0.999774


The first three are right, including the hard ones. *Not bad at all* is positive, which a Bag-of-Words model counting *bad* would get wrong, and *Great, my card was declined again* is correctly read as negative despite *great*.

The last one shows two limits. The model only knows two labels, so *okay* has to be positive or negative. And it says POSITIVE with a score of 0.9998, as sure as it is about everything else. The score is how confident the model is, not how positive the text is, and models are often confident when they should not be. Always check that a model's labels match your problem: this one was trained on film reviews, not on bank messages, and has no "neutral".

### Named entities

**Named entity recognition (NER)** finds the names of people, places and organisations in a text. `aggregation_strategy="simple"` joins the subword pieces back into whole words:

In [10]:
ner = pipeline("token-classification", model="dslim/distilbert-NER", aggregation_strategy="simple")

text = "Ada Lovelace worked with Charles Babbage in London. Today, Google DeepMind has offices there too."
pd.DataFrame(ner(text))[["entity_group", "word", "score", "start", "end"]]

,entity_group,word,score,start,end
0,PER,Ada Lovelace,0.998585,0,12
1,PER,Charles Babbage,0.998313,25,40
2,LOC,London,0.998147,44,50
3,ORG,Google DeepMind,0.985887,59,74


`PER` is a person, `LOC` a location and `ORG` an organisation (`MISC` is used for everything else, such as nationalities). `start` and `end` are the character positions in the text. NER is used to remove names from documents before sharing them, to link news to companies, or to fill in forms from emails.

### Zero-shot classification

The most flexible pipeline needs no training at all. A **zero-shot** classifier takes any text and any list of labels that you invent on the spot, and says which label fits best:

In [11]:
zero_shot = pipeline("zero-shot-classification", model="MoritzLaurer/deberta-v3-xsmall-zeroshot-v1.1-all-33")

result = zero_shot("I have not received my refund yet.",
                   candidate_labels=["payments", "travel advice", "bug report"])
dict(zip(result["labels"], np.round(result["scores"], 2)))

{'payments': np.float64(0.57),
 'bug report': np.float64(0.27),
 'travel advice': np.float64(0.16)}

How can it know labels it has never seen? It was fine-tuned on a different task, **natural language inference**: given two sentences, does the first imply the second? For each label, the pipeline asks whether *I have not received my refund yet* implies *This example is about payments*, and turns the answers into scores that add up to 1.

Try it on the forum posts from Days 1 and 2. They were labelled by forum, but the zero-shot model never sees those labels; it only gets four plain-English descriptions:

In [12]:
categories = ["comp.graphics", "rec.sport.hockey", "sci.space", "talk.politics.mideast"]
news = fetch_20newsgroups(subset="all", categories=categories, remove=("headers", "footers", "quotes"), random_state=0)
posts = pd.DataFrame({"text": news.data, "topic": [news.target_names[t] for t in news.target]})
posts = posts[posts["text"].str.strip().str.len() > 20].reset_index(drop=True)

y = posts["topic"].to_numpy()
train_idx, test_idx = train_test_split(np.arange(len(posts)), test_size=0.25, stratify=y, random_state=0)
print("Train:", len(train_idx), " test:", len(test_idx))

descriptions = {"comp.graphics": "computer graphics", "rec.sport.hockey": "ice hockey",
                "sci.space": "space exploration", "talk.politics.mideast": "Middle East politics"}

Train: 2836  test: 946


This is the same split as on Days 1 and 2, so the numbers can be compared. Each post is cut to its first 500 characters to keep things fast:

In [13]:
test_texts = [" ".join(t.split())[:500] for t in posts["text"].iloc[test_idx]]

start = time.time()
results = zero_shot(test_texts, candidate_labels=list(descriptions.values()), batch_size=16)
print(f"{len(test_texts)} posts in {time.time() - start:.0f} seconds")

label_to_topic = {v: k for k, v in descriptions.items()}
zero_shot_pred = [label_to_topic[r["labels"][0]] for r in results]
print(f"Zero-shot macro F1: {f1_score(y[test_idx], zero_shot_pred, average='macro'):.3f}")

946 posts in 33 seconds
Zero-shot macro F1: 0.899


With **zero** labelled examples, the model gets a macro F1 of 0.90. That is a remarkable starting point when you have no labels at all, for example to sort a pile of support tickets into categories that the business just invented. It is also slow, because the model runs once for every post and every label, and you will now see that a little labelled data does better.

## 6. Fine-tuning DistilBERT

**Fine-tuning** means taking the pretrained model, adding a small new layer on top that outputs your classes, and training the whole thing for a few passes over your labelled data. The pretrained weights already know the language, so they only need small adjustments.

[visual: DistilBERT, pretrained, with its [CLS] vector feeding a new small layer with 4 outputs (graphics, hockey, space, Middle East); arrows showing that training updates both the new layer and, slightly, the pretrained layers]

To keep this fast on a laptop, train on **100 posts per forum** (400 in total) and cut each post to 128 tokens. Day 2 showed that with this few labels, pretrained knowledge makes a big difference.

In [14]:
rng = np.random.RandomState(0)
small_idx = np.concatenate([rng.choice(train_idx[y[train_idx] == t], 100, replace=False) for t in categories])

labels = sorted(categories)
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

def to_dataset(idx):
    data = datasets.Dataset.from_dict({"text": posts["text"].iloc[idx].tolist(),
                                       "label": [label2id[t] for t in y[idx]]})
    return data.map(lambda batch: tokenizer(batch["text"], truncation=True, max_length=128), batched=True)

train_data, test_data = to_dataset(small_idx), to_dataset(test_idx)
train_data

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 400
})

A Hugging Face `Dataset` is like a DataFrame that the training tools understand. `map` runs the tokenizer on every post, so each row now has `input_ids` and an `attention_mask` next to its text and label. Labels must be numbers, hence `label2id`.

`AutoModelForSequenceClassification` loads DistilBERT with a new classification layer for 4 classes. That layer starts with random weights, which is why the model must be trained before use:

In [15]:
classifier = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=len(labels), id2label=id2label, label2id=label2id)

def compute_metrics(prediction):
    predicted = prediction.predictions.argmax(axis=-1)
    return {"macro_f1": f1_score(prediction.label_ids, predicted, average="macro")}

args = TrainingArguments(
    output_dir="forum-classifier",   # where checkpoints would go (none are saved here)
    num_train_epochs=4,              # passes over the 400 posts
    per_device_train_batch_size=16,
    learning_rate=5e-5,              # small: only adjust the pretrained weights
    eval_strategy="epoch",           # score on the test posts after every epoch
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none",
    disable_tqdm=True,
    dataloader_pin_memory=False,
    seed=0,
)
trainer = Trainer(model=classifier, args=args, train_dataset=train_data, eval_dataset=test_data,
                  processing_class=tokenizer, compute_metrics=compute_metrics)

`TrainingArguments` holds the settings, and `Trainer` runs the training loop that you would otherwise write by hand: batches, the forward pass, the loss, the gradients and the weight updates, plus an evaluation after each epoch. `processing_class=tokenizer` lets the trainer pad each batch. The **learning rate** is small on purpose: large updates would destroy what the model learned during pretraining.

Now train. This is the slow cell: one to four minutes on a laptop CPU, a few seconds on a GPU.

In [16]:
start = time.time()
trainer.train()
print(f"Training took {time.time() - start:.0f} seconds")

history = pd.DataFrame(trainer.state.log_history)
history.groupby("epoch").first()[["loss", "eval_loss", "eval_macro_f1"]].round(3)

{'loss': '0.9335', 'grad_norm': '2.385', 'learning_rate': '3.8e-05', 'epoch': '1'}


{'eval_loss': '0.42', 'eval_macro_f1': '0.9092', 'eval_runtime': '3.458', 'eval_samples_per_second': '273.6', 'eval_steps_per_second': '34.42', 'epoch': '1'}


{'loss': '0.2459', 'grad_norm': '5.85', 'learning_rate': '2.55e-05', 'epoch': '2'}


{'eval_loss': '0.2712', 'eval_macro_f1': '0.9116', 'eval_runtime': '3.432', 'eval_samples_per_second': '275.7', 'eval_steps_per_second': '34.67', 'epoch': '2'}


{'loss': '0.08398', 'grad_norm': '8.125', 'learning_rate': '1.3e-05', 'epoch': '3'}


{'eval_loss': '0.285', 'eval_macro_f1': '0.913', 'eval_runtime': '3.398', 'eval_samples_per_second': '278.4', 'eval_steps_per_second': '35.02', 'epoch': '3'}


{'loss': '0.03944', 'grad_norm': '0.4416', 'learning_rate': '5e-07', 'epoch': '4'}


{'eval_loss': '0.2687', 'eval_macro_f1': '0.9115', 'eval_runtime': '3.4', 'eval_samples_per_second': '278.2', 'eval_steps_per_second': '35', 'epoch': '4'}
{'train_runtime': '46.2', 'train_samples_per_second': '34.63', 'train_steps_per_second': '2.164', 'train_loss': '0.3257', 'epoch': '4'}
Training took 47 seconds


,loss,eval_loss,eval_macro_f1
epoch,,,
1.0,0.934,0.420,0.909
2.0,0.246,0.271,0.912
3.0,0.084,0.285,0.913
4.0,0.039,0.269,0.912


The macro F1 on the 946 test posts is already 0.91 after the first epoch and barely moves after that. The training loss keeps falling, to almost zero, while the test loss stops improving after the second epoch: from there on, the model is mostly memorising the 400 posts (overfitting, as in Week 4). Watching the test loss is how you choose the number of epochs.

The trained model works like any other in a `pipeline`. Call `.eval()` first: it switches off **dropout**, the random switching-off of neurons that helps during training but must not happen when predicting.

In [17]:
trainer.model.eval()                     # switch off dropout before predicting
forum_classifier = pipeline("text-classification", model=trainer.model, tokenizer=tokenizer)
forum_classifier(["The goalie was injured in the third period.",
                  "What file format should I use to save a rendered 3D image?",
                  "The launch was delayed because of a problem with the fuel tank."])

[{'label': 'rec.sport.hockey', 'score': 0.9741339087486267},
 {'label': 'comp.graphics', 'score': 0.9774437546730042},
 {'label': 'sci.space', 'score': 0.9720311164855957}]

### Which method wins?

Put the fine-tuned model next to the methods of Days 1 and 2, all trained on the same 400 posts and tested on the same 946:

In [18]:
texts = posts["text"].to_numpy()

tfidf_model = make_pipeline(TfidfVectorizer(stop_words="english"), LogisticRegression(max_iter=1000))
tfidf_model.fit(texts[small_idx], y[small_idx])

minilm = SentenceTransformer("all-MiniLM-L6-v2")
train_vectors = minilm.encode(texts[small_idx].tolist(), normalize_embeddings=True)
test_vectors = minilm.encode(texts[test_idx].tolist(), normalize_embeddings=True)
embedding_model = LogisticRegression(max_iter=2000).fit(train_vectors, y[small_idx])

fine_tuned_pred = trainer.predict(test_data).predictions.argmax(axis=-1)

comparison = pd.DataFrame({
    "labelled posts": [0, 400, 400, 400],
    "macro F1": [f1_score(y[test_idx], zero_shot_pred, average="macro"),
                 f1_score(y[test_idx], tfidf_model.predict(texts[test_idx]), average="macro"),
                 f1_score(y[test_idx], embedding_model.predict(test_vectors), average="macro"),
                 f1_score(y[test_idx], [id2label[i] for i in fine_tuned_pred], average="macro")],
}, index=["Zero-shot (section 5)", "TF-IDF + logistic regression (Day 1)",
          "Embeddings + logistic regression (Day 2)", "Fine-tuned DistilBERT (today)"])
comparison.round(3)

,labelled posts,macro F1
Zero-shot (section 5),0,0.899
TF-IDF + logistic regression (Day 1),400,0.901
Embeddings + logistic regression (Day 2),400,0.939
Fine-tuned DistilBERT (today),400,0.912


The fine-tuned DistilBERT (0.91) beats TF-IDF (0.90), but **yesterday's embeddings with a logistic regression win** (0.94), and the zero-shot model, with no labels at all, is almost as good as TF-IDF with 400. That is a useful lesson, not a failure. The forums use very different words, so the topic is easy to spot. MiniLM was trained specifically to make good sentence vectors, while DistilBERT had to learn the task from 400 posts that were cut to 128 tokens. And the logistic regression trains in a second, against a minute for the fine-tuning.

Fine-tuning pays off when the task depends on **how** things are said, not just on which topic words appear: sentiment, intent, tone, or classes that use the same vocabulary. The lab tests exactly that, on bank messages whose intents share most of their words.

The practical rule is to try the simple methods first. Use zero-shot when you have no labels, TF-IDF or embeddings with a linear model as your baseline, and fine-tune when the baseline is not good enough and you have a few hundred labelled examples.

## 7. Generative models: predicting the next token

The second family of transformers, the **decoders**, is pretrained on a different game: read the text so far, and predict the **next token**. GPT, Llama, Claude and Gemini are all decoders. The slides call it a *universal task*: almost any task can be written as text to continue, such as *Is this text positive or negative? ...* or *A summary of the text above:*.

[visual: "The capital of France is" → GPT → probabilities for the next token (Paris 0.45, the 0.27, ...) → append "Paris" → repeat]

**SmolLM2-135M-Instruct** is a tiny decoder (135 million parameters, about 270 MB) published by Hugging Face. Look at the probabilities it gives for the next token:

In [19]:
generator_name = "HuggingFaceTB/SmolLM2-135M-Instruct"
gen_tokenizer = AutoTokenizer.from_pretrained(generator_name)
gen_model = AutoModelForCausalLM.from_pretrained(generator_name)

def next_token_probabilities(text, k=5):
    inputs = gen_tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        logits = gen_model(**inputs).logits[0, -1]        # scores for the token after the last one
    probabilities = torch.softmax(logits, dim=-1)
    top = torch.topk(probabilities, k)
    return [(gen_tokenizer.decode(int(i)), round(float(p), 3)) for p, i in zip(top.values, top.indices)]

for text in ["The capital of France is", "My card has not arrived, so I"]:
    print(text, next_token_probabilities(text))

The capital of France is [(' Paris', 0.445), (' the', 0.27), (' located', 0.088), (' called', 0.021), (' Le', 0.015)]
My card has not arrived, so I [(' will', 0.188), ("'ll", 0.165), (' need', 0.113), ("'m", 0.078), (' have', 0.047)]


*Paris* is the most likely next token, but only at 0.45: *the*, *located* and *called* are also possible continuations (*The capital of France is the city of...*). Generating text is just this, repeated: pick a token, add it to the text, and predict again.

**How the next token is picked** matters a lot:

+ **Greedy** decoding always picks the most likely token. It is predictable, and the same input always gives the same output.
+ **Sampling** draws a token at random according to the probabilities. The **temperature** controls how adventurous it is: below 1 it favours likely tokens, above 1 it gives unlikely tokens more chances. You will set it in the OpenAI API tomorrow.

The `text-generation` pipeline does the loop for you. *Instruct* models were fine-tuned further to follow instructions in a chat format, so you pass a list of messages:

In [20]:
chat = pipeline("text-generation", model=gen_model, tokenizer=gen_tokenizer)

article = ("Our new mobile app lets customers freeze a lost card in two taps. Since the launch in March, "
           "calls to the support line about lost cards fell by 40 percent, and the average time to block a card "
           "dropped from nine minutes to under one. The bank now plans to add the same feature to its website.")
messages = [{"role": "user", "content": "Summarise in one sentence:\n" + article}]
reply = chat(messages, max_new_tokens=60, do_sample=False)
print(reply[0]["generated_text"][-1]["content"])

The new mobile app helps customers freeze lost cards in two taps.


A correct sentence, from a model small enough to run anywhere. It is not a great summary, though: it drops the two numbers (40 percent fewer calls, under one minute to block a card) that are the actual news. Now sample three answers to the same question with a high temperature:

In [21]:
torch.manual_seed(0)
messages = [{"role": "user", "content": "Give me one name for a bank's new savings app."}]
for _ in range(3):
    reply = chat(messages, max_new_tokens=20, do_sample=True, temperature=1.2)
    print("-", reply[0]["generated_text"][-1]["content"])

- The age-old question of how my bank should choose its new app. We have just implemented a
- A suitable name for a bank's savings app might be The My Hero. Based on this option,


- One name for such a bank's innovative savings app is BankBankPal, which is currently launched on


Three different answers, because each run draws different tokens. That is what you want for brainstorming, and not what you want for a classifier or a summary that must be reproducible. Notice also that none of the three is a good, short app name: a model this small follows instructions loosely, rambles, and is cut off at 20 tokens.

Small models are fast and private, but they make mistakes that larger ones do not:

In [22]:
messages = [{"role": "user", "content": "Translate to Spanish: Where is my new card? I ordered it two weeks ago."}]
print(chat(messages, max_new_tokens=40, do_sample=False)[0]["generated_text"][-1]["content"])

Encontrarás la vista de la vida, este es un idiomatico que puede ser traduciendo al español.


The "translation" is not what the customer said. A 135-million-parameter model has not seen enough Spanish to translate reliably, and it gives no warning that it got it wrong: the text looks fluent either way. This is the most important thing to remember about generative models of every size: **fluent is not the same as correct**.

The large models you use tomorrow, through the OpenAI API, have hundreds of times more parameters and get tasks like this right. They run on someone else's servers, so you pay for each call and send them your text. The choice between a small local model and a large remote one is about quality, cost and privacy, and you will make it often.

## 8. Choosing a model responsibly

The Hub makes it easy to download a model. Before you use one for real, read its **model card**, the page that describes it on the Hub. Check five things:

1. **Task and training data.** What was it fine-tuned on? The sentiment model above learned from film reviews, so it has no idea that *pending* is bad news in a bank.
2. **Language.** Most of today's models are English only.
3. **Size and speed.** DistilBERT runs on a laptop, larger models need a GPU.
4. **Licence.** Many models allow commercial use (Apache 2.0, MIT). Some do not, and some datasets have their own conditions.
5. **Known limits and biases.** Good model cards list them, and you should test them yourself.

The last point deserves a test. Here is the pretrained DistilBERT again, filling in a job:

In [23]:
for sentence in ["She works as a [MASK] in a hospital.", "He works as a [MASK] in a hospital."]:
    print(sentence, [(g["token_str"], round(g["score"], 2)) for g in fill(sentence, top_k=4)])

She works as a [MASK] in a hospital. [('nurse', 0.83), ('receptionist', 0.05), ('doctor', 0.01), ('volunteer', 0.01)]
He works as a [MASK] in a hospital. [('nurse', 0.35), ('doctor', 0.16), ('physician', 0.07), ('clerk', 0.03)]


For *she*, the model puts 0.83 on *nurse* and only 0.01 on *doctor*. For *he*, *nurse* drops to 0.35 and *doctor* rises to 0.16. The model learned this from the books and Wikipedia pages it was trained on, like GloVe yesterday, and a model fine-tuned from it starts with the same associations.

When a model helps decide something about people, such as screening CVs, flagging transactions or routing complaints, test it with inputs that differ only in a name, a gender or a nationality, and check that the result does not change. If it does, do not ship it.

Two more practical points. Models from the Hub run on your own computer, so your data never leaves it, which matters for customer messages and medical or legal text. And training uses energy: fine-tuning DistilBERT on a laptop costs very little, but pretraining a large model costs as much electricity as hundreds of homes use in a year, which is one more reason to reuse pretrained models instead of training from scratch.

## Summary

| Idea | What it means | Tool |
|---|---|---|
| Transformer | network that builds each token's vector by attending to all the others | |
| Tokenizer | splits text into subword tokens and maps them to ids | `AutoTokenizer.from_pretrained(name)` |
| Attention mask, padding | mark real tokens (1) and padding (0) so batches have equal lengths | `tokenizer(texts, padding=True, truncation=True)` |
| Contextual vectors | the same word gets a different vector in each sentence | `AutoModel`, `last_hidden_state` |
| Masked language modelling | encoder pretraining: guess the hidden words | `pipeline("fill-mask")` |
| Pipeline | tokenizer + model + post-processing in one call | `pipeline("sentiment-analysis")`, `"token-classification"`, `"zero-shot-classification"` |
| Zero-shot | classify into labels you invent, with no training | `zero_shot(text, candidate_labels=[...])` |
| Fine-tuning | train a pretrained model with a new head on your labels | `AutoModelForSequenceClassification`, `TrainingArguments`, `Trainer` |
| Next-token prediction | decoder pretraining; generation repeats it | `AutoModelForCausalLM`, `pipeline("text-generation")` |
| Temperature | how random the choice of the next token is | `do_sample=True, temperature=...` |

The rules that matter most:

+ **Always use a model's own tokenizer**, and remember its maximum length.
+ **Try the simple methods first.** Zero-shot with no labels, TF-IDF or embeddings as a baseline, fine-tuning when they are not enough.
+ **A confident score is not a correct answer**, and fluent text is not a correct answer either.
+ **Read the model card** and test for bias before a model makes decisions about people.

In the lab, you route a bank's customer messages with a zero-shot model and a fine-tuned DistilBERT, and find out whether fine-tuning beats yesterday's semantic search.

## Appendix

### A.1 Saving and sharing a fine-tuned model

The fine-tuned model above only lives in memory. To keep it, save the model and its tokenizer to a folder (about 270 MB), and load them back with the same `from_pretrained` you used for the Hub:

```python
trainer.save_model("forum-classifier")          # model weights and configuration
tokenizer.save_pretrained("forum-classifier")
reloaded = pipeline("text-classification", model="forum-classifier")
```

With a free Hugging Face account, `trainer.push_to_hub()` uploads it to the Hub so that others can use it with its name. Do not commit model folders to git: they are large, and GitHub rejects files over 100 MB.

### A.2 Training on a GPU

`Trainer` uses a GPU automatically when PyTorch finds one (NVIDIA with CUDA, or Apple silicon). On a laptop without one, [Google Colab](https://colab.research.google.com) gives a free GPU: open the notebook there, choose **Runtime → Change runtime type → T4 GPU**, and the fine-tuning in section 6 takes a few seconds instead of minutes. This is the way to fine-tune on all 3,000 training posts, or with longer texts.

In [24]:
print("CUDA GPU available:", torch.cuda.is_available())
print("Apple GPU available:", torch.backends.mps.is_available())

CUDA GPU available: False
Apple GPU available: True


### A.3 Looking at attention

You can ask the model for its attention weights with `output_attentions=True`. The fast attention code that `transformers` uses by default does not keep the weights, so load the model with `attn_implementation="eager"`, the slower version that does. For each layer, you get a tensor of shape (batch, heads, tokens, tokens): every attention **head** gives, for every token, how much it looks at every other token. Here is where *bank* looks in the last layer, averaged over the heads:

In [25]:
eager_encoder = AutoModel.from_pretrained(checkpoint, attn_implementation="eager")
inputs = tokenizer("We had a picnic on the river bank.", return_tensors="pt")
with torch.no_grad():
    attentions = eager_encoder(**inputs, output_attentions=True).attentions

print("Layers:", len(attentions), " shape of one layer:", tuple(attentions[-1].shape))
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
bank = tokens.index("bank")
weights = attentions[-1][0, :, bank, :].mean(dim=0)
pd.Series(weights.numpy().round(3), index=tokens).sort_values(ascending=False).head(5)

Layers: 6  shape of one layer: (1, 12, 11, 11)


.        0.462
[SEP]    0.296
river    0.045
bank     0.036
[CLS]    0.035
dtype: float32

Most of the attention goes to the full stop and `[SEP]`, and *river* is only third, although it is the word that decides the meaning of *bank*. That is typical: attention weights are a hint, not an explanation. Each layer and head attends differently, a lot of attention goes to special tokens and punctuation, and research has shown that high attention does not always mean that a word mattered for the result.

### A.4 More pipelines

The same `pipeline` function covers many other tasks: `"question-answering"`-style models for documents, `"automatic-speech-recognition"` (for example `openai/whisper-tiny`), `"image-classification"`, `"zero-shot-image-classification"` (CLIP), and more. `from transformers.pipelines import SUPPORTED_TASKS; sorted(SUPPORTED_TASKS)` lists them all. The [task pages on the Hub](https://huggingface.co/tasks) show which models fit each task.